# TP - IRVE

Travail realisé par :
- Issam Belhorma
- Zakaria Soual

In [1]:
%pip install -r requirements.txt

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached appnope-0.1.4-py2.py3-none-any.whl.metadata (908 bytes)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
  Using cached charset_normalizer-3.4.7-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached dacite-1.9.2-py3-none-any.whl.metadata (17 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached fonttools-4.63.0-cp313-cp313-win_amd64.whl.metadata (121 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached ImageHash-4.3.2-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached joblib-1.5.3-py3-none-a


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Partie 1 - Profiling & audit

- charger le CSV consolide depuis `data.gouv.fr` ;
- generer un rapport `ydata-profiling` dans le notebook ;
- remplir une petite grille d'audit avec les problemes principaux ;
- calculer un score de qualite global initial.

### Chargement des donnees

On charge directement l'URL officielle, parce que le sujet demande explicitement de travailler sur le fichier consolide publie sur `data.gouv.fr`.

In [3]:
import numpy as np
import pandas as pd
from ydata_profiling import ProfileReport

CSV_URL = 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435'

# low_memory=False permet d'avoir des types plus stables, ce qui est preferable pour un audit.
df = pd.read_csv(CSV_URL, sep=',', low_memory=False)
df_checkpoint = df.copy()  

print(f"Nombre de lignes : {df.shape[0]:,}".replace(',', ' '))
print(f'Nombre de colonnes : {df.shape[1]}')

df.head()

Nombre de lignes : 227 232
Nombre de colonnes : 52


,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
0,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004017,ATHTBE1004017,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
1,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004018,ATHTBE1004018,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
2,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004019,ATHTBE1004019,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
3,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004020,ATHTBE1004020,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
4,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,HAG_P22_Slave_3,ATHTBE1004957,ATHTBE1004957,HAG_P22_Slave_3,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.761841,48.827040,NaN,NaN,False,False,False


In [11]:
df = df_checkpoint.copy() 

### Rapport de profiling automatique

Le profiling automatique sert de premier diagnostic : il donne une vue d'ensemble sur les 52 colonnes avant l'audit manuel.

In [12]:
profile = ProfileReport(
    df,
    title='Partie 1 - Profiling initial IRVE',
    explorative=True,
    minimal=True,
)

profile.to_file("rapport_qualite_donnees-irve.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 52/52 [00:06<00:00,  8.02it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
for c in df.columns:
    print(f"=== Colonne : {c} ===")
    
    # 1. Type de données et taux de complétude
    total_rows = len(df)
    missing_values = df[c].isna().sum()
    missing_percentage = (missing_values / total_rows) * 100
    distinct_values = df[c].nunique()
    
    print(f"• Type : {df[c].dtype}")
    print(f"• Valeurs manquantes : {missing_values} ({missing_percentage:.2f}%)")
    print(f"• Valeurs uniques (cardinalité) : {distinct_values}")
    
    # 2. Statistiques selon le type de colonne
    if pd.api.types.is_numeric_dtype(df[c]):
        # Pour les colonnes numériques (ex: coordonnées, puissance des bornes)
        print(f"• Minimum : {df[c].min()}")
        print(f"• Maximum : {df[c].max()}")
        print(f"• Moyenne : {df[c].mean():.2f}")
        print(f"• Médiane : {df[c].median():.2f}")
    else:
        # Pour les colonnes textuelles / catégorielles (ex: communes, types de prises)
        print("• Top 5 des valeurs les plus fréquentes :")
        top_5 = df[c].value_counts(dropna=False).head(5)
        for val, count in top_5.items():
            pct = (count / total_rows) * 100
            print(f"  - {val}: {count} ({pct:.2f}%)")
            
    print("\n" + "-"*40 + "\n")

=== Colonne : nom_amenageur ===
• Type : object
• Valeurs manquantes : 1317 (0.58%)
• Valeurs uniques (cardinalité) : 4408
• Top 5 des valeurs les plus fréquentes :
  - Power Dot France: 13971 (6.15%)
  - TotalEnergies Marketing France: 8011 (3.53%)
  - GROUPE INDIGO: 7415 (3.26%)
  - DRIVECO: 7007 (3.08%)
  - Electra: 5619 (2.47%)

----------------------------------------

=== Colonne : siren_amenageur ===
• Type : float64
• Valeurs manquantes : 90985 (40.04%)
• Valeurs uniques (cardinalité) : 2940
• Minimum : 0.0
• Maximum : 992163725.0
• Moyenne : 692420034.80
• Médiane : 842718512.00

----------------------------------------

=== Colonne : contact_amenageur ===
• Type : object
• Valeurs manquantes : 72349 (31.84%)
• Valeurs uniques (cardinalité) : 908
• Top 5 des valeurs les plus fréquentes :
  - nan: 72349 (31.84%)
  - hello@powerdot.fr: 13971 (6.15%)
  - supervision-ev.france@totalenergies.com: 12160 (5.35%)
  - contact@lidl.fr: 10573 (4.65%)
  - support@driveco.com: 7007 (3.08%)

### Audit simple des problemes principaux

On commence par regarder la completude de toutes les colonnes, puis on construit une petite grille d'audit sur quelques problemes tres visibles du fichier :

- codes postaux manquants ;
- dates de mise en service manquantes ;
- puissances aberrantes ;
- identifiants repetes ;
- coordonnees signalees comme douteuses ;
- dates tres anciennes.

In [14]:
# Cette table donne une vue simple de la completude sur les 52 colonnes.
completude = pd.DataFrame({
    'colonne': df.columns,
    'nb_manquants': df.isna().sum().values,
    'pct_manquants': (df.isna().mean() * 100).round(2).values,
})

completude = completude.sort_values('pct_manquants', ascending=False).reset_index(drop=True)
completude

,colonne,nb_manquants,pct_manquants
0,observations,180712,79.53
1,tarification,170887,75.20
2,num_pdl,111260,48.96
3,cable_t2_attache,106741,46.97
4,consolidated_code_postal,92870,40.87
5,date_mise_en_service,92271,40.61
6,siren_amenageur,90985,40.04
7,raccordement,79307,34.90
8,id_pdc_local,78678,34.62
9,consolidated_commune,76393,33.62


In [15]:
date_maj = pd.to_datetime(df['date_maj'], errors='coerce')
date_mise_en_service = pd.to_datetime(df['date_mise_en_service'], errors='coerce')
puissance_nominale = pd.to_numeric(df['puissance_nominale'], errors='coerce')


# Petite fonction de gravite simple pour rendre la grille plus lisible.
def gravite(pct):
    if pct >= 40:
        return 'Critique'
    if pct >= 20:
        return 'Elevee'
    if pct >= 5:
        return 'Moyenne'
    return 'Faible'


# Grille Audit
grille_audit = pd.DataFrame([
    {
        'colonne': 'consolidated_code_postal',
        'dimension': 'Completude',
        'probleme': 'Code postal manquant',
        'pct_problemes': round(df['consolidated_code_postal'].isna().mean() * 100, 2),
    },
    {
        'colonne': 'date_mise_en_service',
        'dimension': 'Completude',
        'probleme': 'Date de mise en service manquante',
        'pct_problemes': round(df['date_mise_en_service'].isna().mean() * 100, 2),
    },
    {
        'colonne': 'puissance_nominale',
        'dimension': 'Validite',
        'probleme': 'Puissance <= 0 ou > 1000 kW',
        'pct_problemes': round(((puissance_nominale <= 0) | (puissance_nominale > 1000)).mean() * 100, 2),
    },
    {
        'colonne': 'id_pdc_itinerance',
        'dimension': 'Unicite',
        'probleme': 'Identifiant repete',
        'pct_problemes': round((df['id_pdc_itinerance'].duplicated(keep=False)).mean() * 100, 2),
    },
    {
        'colonne': 'consolidated_is_lon_lat_correct',
        'dimension': 'Exactitude',
        'probleme': 'Coordonnees signalees comme douteuses',
        'pct_problemes': round((df['consolidated_is_lon_lat_correct'] == False).mean() * 100, 2),
    },
    {
        'colonne': 'date_mise_en_service',
        'dimension': 'Coherence',
        'probleme': 'date_mise_en_service > date_maj',
        'pct_problemes': round((
            (date_mise_en_service.notna())
            & (date_maj.notna())
            & (date_mise_en_service > date_maj)
        ).mean() * 100, 2),
    },
    {
        'colonne': 'date_maj',
        'dimension': 'Fraicheur',
        'probleme': 'Date de mise a jour a plus de 365 jours du maximum du fichier',
        'pct_problemes': round((date_maj < (date_maj.max() - pd.Timedelta(days=365))).mean() * 100, 2),
    },
])

grille_audit['gravite'] = grille_audit['pct_problemes'].apply(gravite)
grille_audit = grille_audit.sort_values('pct_problemes', ascending=False).reset_index(drop=True)

grille_audit

,colonne,dimension,probleme,pct_problemes,gravite
0,id_pdc_itinerance,Unicite,Identifiant repete,57.63,Critique
1,consolidated_code_postal,Completude,Code postal manquant,40.87,Critique
2,date_mise_en_service,Completude,Date de mise en service manquante,40.61,Critique
3,consolidated_is_lon_lat_correct,Exactitude,Coordonnees signalees comme douteuses,35.97,Elevee
4,date_maj,Fraicheur,Date de mise a jour a plus de 365 jours du max...,35.63,Elevee
5,puissance_nominale,Validite,Puissance <= 0 ou > 1000 kW,2.75,Faible
6,date_mise_en_service,Coherence,date_mise_en_service > date_maj,1.07,Faible


In [16]:
# On calcule un score simple par dimension : 100 - pourcentage moyen de problemes sur la dimension.
def score_dimension(series):
    if len(series) == 0:
        return 100.0
    return round(100 - series.mean(), 2)


scores = pd.DataFrame([
    {'dimension': 'Completude', 'score_initial': score_dimension(completude['pct_manquants'])},
    {'dimension': 'Validite', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Validite', 'pct_problemes'])},
    {'dimension': 'Coherence', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Coherence', 'pct_problemes'])},
    {'dimension': 'Unicite', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Unicite', 'pct_problemes'])},
    {'dimension': 'Exactitude', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Exactitude', 'pct_problemes'])},
    {'dimension': 'Fraicheur', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Fraicheur', 'pct_problemes'])},
])

score_global_initial = round(scores['score_initial'].mean(), 2)

display(scores)
print(f'Score de qualite global initial : {score_global_initial}/100')

,dimension,score_initial
0,Completude,87.73
1,Validite,97.25
2,Coherence,98.93
3,Unicite,42.37
4,Exactitude,64.03
5,Fraicheur,64.37


Score de qualite global initial : 75.78/100


 # Partie 2

### Remise a zero du dataset

On prends le dataset du début

In [10]:
df_partie2 = df_checkpoint.copy()
df_partie2.head()

,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
0,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004017,ATHTBE1004017,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
1,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004018,ATHTBE1004018,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
2,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004019,ATHTBE1004019,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
3,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004020,ATHTBE1004020,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
4,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,HAG_P22_Slave_3,ATHTBE1004957,ATHTBE1004957,HAG_P22_Slave_3,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.761841,48.827040,NaN,NaN,False,False,False


### Etude du grain

On cherche ici a comprendre si une ligne represente un point de charge unique, ou un point de charge observe dans plusieurs livraisons.

In [11]:
grain_resume = pd.DataFrame([
    {'indicateur': 'Nombre total de lignes', 'valeur': len(df_partie2)},
    {'indicateur': "Nombre d'id_pdc_itinerance non nuls", 'valeur': df_partie2['id_pdc_itinerance'].notna().sum()},
    {'indicateur': "Nombre d'id_pdc_itinerance uniques", 'valeur': df_partie2['id_pdc_itinerance'].nunique(dropna=True)},
    {'indicateur': 'Nombre de lignes avec un id repete', 'valeur': df_partie2['id_pdc_itinerance'].duplicated(keep=False).sum()},
])

grain_resume

,indicateur,valeur
0,Nombre total de lignes,227232
1,Nombre d'id_pdc_itinerance non nuls,227232
2,Nombre d'id_pdc_itinerance uniques,160138
3,Nombre de lignes avec un id repete,130964


In [ ]:
# On affiche quelques exemples concrets pour voir pourquoi un meme id peut revenir plusieurs fois.
exemples_repetes = (
    df_partie2[df_partie2['id_pdc_itinerance'].duplicated(keep=False)]
    [['id_pdc_itinerance', 'nom_station', 'date_maj', 'last_modified', 'datagouv_resource_id']]
    .sort_values(['id_pdc_itinerance', 'date_maj', 'last_modified'])
    .head(20)
)

exemples_repetes

### Regle de dedoublonnage retenue

Choix retenu : garder la livraison la plus recente pour chaque `id_pdc_itinerance`.

Justification simple :
- on veut conserver une version la plus recente de chaque point de charge ;
- on trie par `date_maj` decroissante ;
- en cas d'egalite, on utilise `last_modified` puis `created_at`.

In [ ]:
df_dedup_work = df_partie2.copy()
df_dedup_work['date_maj_dt'] = pd.to_datetime(df_dedup_work['date_maj'], errors='coerce')
df_dedup_work['last_modified_dt'] = pd.to_datetime(df_dedup_work['last_modified'], errors='coerce')
df_dedup_work['created_at_dt'] = pd.to_datetime(df_dedup_work['created_at'], errors='coerce')

# On ne peut pas dedoublonner proprement les lignes sans identifiant stable.
avec_id = df_dedup_work[df_dedup_work['id_pdc_itinerance'].notna()].copy()
sans_id = df_dedup_work[df_dedup_work['id_pdc_itinerance'].isna()].copy()

avec_id_dedup = (
    avec_id
    .sort_values(
        ['id_pdc_itinerance', 'date_maj_dt', 'last_modified_dt', 'created_at_dt'],
        ascending=[True, False, False, False],
        na_position='last'
    )
    .drop_duplicates(subset='id_pdc_itinerance', keep='first')
)

df_dedoublonne = pd.concat([avec_id_dedup, sans_id], ignore_index=True)
df_dedoublonne = df_dedoublonne.drop(columns=['date_maj_dt', 'last_modified_dt', 'created_at_dt'])

df_dedoublonne.head()

### Impact du dedoublonnage

On compare simplement le nombre de lignes avant et apres.

In [ ]:
impact_dedoublonnage = pd.DataFrame([
    {'indicateur': 'Nombre de lignes avant dedoublonnage', 'valeur': len(df_partie2)},
    {'indicateur': 'Nombre de lignes apres dedoublonnage', 'valeur': len(df_dedoublonne)},
    {'indicateur': 'Nombre de lignes retirees', 'valeur': len(df_partie2) - len(df_dedoublonne)},
    {'indicateur': "Nombre d'id_pdc_itinerance uniques avant", 'valeur': df_partie2['id_pdc_itinerance'].nunique(dropna=True)},
    {'indicateur': "Nombre d'id_pdc_itinerance uniques apres", 'valeur': df_dedoublonne['id_pdc_itinerance'].nunique(dropna=True)},
])

impact_dedoublonnage